<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
DeepAgents - Meeting/Briefing-Skill
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der PMO-Assistent unterstützt Meeting-Vorbereitung, Aufgabenverfolgung und Statusreporting. Dieses Notebook bündelt die Verarbeitung von Protokollen, Risikolisten und offenen Punkten in einem Skill für Briefings, Aufgaben- und Statusübersichten. Der Koordinator nutzt strukturierte Ergebnisse und einen Writer für die Formulierung. Quellenprüfung, formales Gate und menschliche Freigabe bleiben getrennte Schritte; der Skill trifft keine verbindlichen Projektentscheidungen.

**Empfohlener Lernpfad:** [Harness-Grundlagen](M50_DeepAgents_Harness.ipynb) → [PMO-Skill](M51_DeepAgents_Skill_PMO_Assistent.ipynb) → [mehrere PMO-Skills](M52_DeepAgent_Multi_Skill.ipynb) → [optionale Parameter- und Sandbox-Vertiefung](M53_DeepAgents_Vertiefung.ipynb). Der Lernpfad führt vom Grundaufbau über einen wiederverwendbaren Arbeitsstandard zur Kombination mehrerer Skills. Die Parameter- und Sandbox-Vertiefung ergänzt diesen Ablauf bei Bedarf.

**Praxis-Transfer:** Mara Vogt benötigt aus denselben Projektunterlagen je nach Anlass ein Meeting-Briefing, eine Aufgabenübersicht oder eine Statusübersicht. Der PMO-Assistent nutzt dafür einen wiederverwendbaren Skill. Quellen, offene Angaben und Prüfstatus bleiben sichtbar; ein formales Gate ersetzt keine menschliche Freigabe.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
    copy_from_github,
)

import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"]  = "m51_pmo_briefing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["deepagents"],
    "metadata": {"notebook": "deepagents_skill_pmo_assistent", "version": "1.0"}
}

In [ ]:
#@title 📦 Installationen { display-mode: "form" }

!uv pip install --system -q "deepagents==0.7.34"
!uv pip install --system -q markitdown[all]
# Originaldokumente nach dem Retrieval vollständig konvertieren.
!uv pip install --system -q "markitdown[all]"


In [ ]:
#@title 📂 Google Drive mounten (PMO-RAG) { display-mode: "form" }
from google.colab import drive

drive.mount("/content/drive")

# 1 | Übersicht

---

<div style="background-color:#1a1a2e; padding:30px; border-radius:12px; margin-bottom:10px">
  <h1 style="color:#e0e0e0; font-family:monospace; margin:0">PMO-Briefing-Skill mit DeepAgents</h1>
  <p style="color:#a0a0c0; font-family:monospace; margin:8px 0 0 0">Skill-Dateien aus GitHub · Gate-Analyse mit JUDGE (gpt-6.1-sol) · structured_output · Writer Sub-Agent</p>
</div>

**Was dieses Notebook zeigt:**
- Skill-Dateien (`SKILL.md`, `WRITER.md`, Regelwerke) **direkt aus dem GitHub-Repo laden**
- Gate-Analyse mit `JUDGE` (`gpt-6.1-sol`) + `with_structured_output()` → Pydantic-validiertes JSON
- `extract_actions.py` aus GitHub holen, lokal zwischenspeichern, deterministisch aufrufen
- **Writer als Sub-Agent** mit geladenem `WRITER.md` als System-Prompt
- DeepAgent-Koordinator orchestriert Skill-Abarbeitung autonom

---

<font color="black" size="5">Was dieses Notebook zeigt</font>

In `06_skill/pmo-briefing` im GitHub-Repo liegt ein vollständiger Skill:
Kontext laden, Agenda nach festen Regeln strukturieren, Action Items deterministisch
extrahieren und das Ergebnis im definierten Briefing-Format ausgeben.

Dieses Notebook baut daraus einen **DeepAgent** mit vier Kernmerkmalen:

| # | Merkmal | Warum |
|---|---------|-------|
| 1 | **Skill-Dateien aus GitHub laden** | Änderungen am Skill greifen ohne Notebook-Anpassung |
| 2 | **`with_structured_output()`** | Gate-Analyse gibt Pydantic-validiertes JSON zurück |
| 3 | **Modell-Auswahl laut Guide** | `ROUTER` (`gpt-6-luna`) für den Koordinator, `JUDGE` (`gpt-6.1-sol`) für die Gate-Analyse |
| 4 | **Writer als Sub-Agent** | Trennung von Analyse und Formatierung |

> **Best Practice:** DeepAgents ist hier die Harness-Schicht, nicht die Wissensquelle.
> Die Fachlogik bleibt im Skill-Repo, in den Regelwerken und im deterministischen Tooling verankert.

In [ ]:
#@markdown   <p><font size="4" color="green">Skill-Architektur</font></p>

diagram_arch = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) --> LOAD
    U([" 🧑 Meeting-Anfrage"]) --> K

    subgraph K_BLOCK [" 🤖 DeepAgent Koordinator — ROUTER · gpt-6-luna"]
        LOAD["Skill-Dateien laden\n(copy_from_github)"] --> K
        K["Kontext laden\nAction Items extrahieren"]
        K --> G["🛠️ analyze_meeting_context\nJUDGE · gpt-6.1-sol + with_structured_output"]
        K --> E["🛠️ extract_action_items\nextract_actions.py als Subprocess"]
        G --> JSON["GateOutputResult JSON\nPydantic-validiert"]
        E --> JSON
        JSON --> W["Sub-Agent\nbriefing-writer"]
    end

    W --> B([" ✅ PMO-Briefing"])

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class GH,U input;
    class B output;
    class LOAD,JSON target;
'''
mermaid(diagram_arch, width=580)

# 2 | Skill-Dateien aus GitHub laden

---



Alle Skill-Dateien werden **einmalig aus dem GitHub-Repo** in ein lokales Cache-Verzeichnis geladen:

| Datei | Zweck |
|-------|-------|
| `SKILL.md` | Gate-Regeln, Hard Rules, Workflow-Definition |
| `references/writer-format.md` | Anweisungen für den Writer Sub-Agenten |
| `references/agenda_rules.md` | Prioritäten, Agenda-Aufbau |
| `references/action_rules.md` | Action-Item-Erkennungsregeln |
| `references/examples.md` | Referenzbeispiele für Gate und Writer |
| `scripts/extract_actions.py` | Deterministisches Skript — direkt im Cache-Verzeichnis |

> 💡 Das Skript `extract_actions.py` liegt nach dem Download direkt im Cache-Verzeichnis
> und wird per `subprocess` aufgerufen — kein Temp-Dateien-Schreiben nötig.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path
from textwrap import dedent
from typing import List, Optional, Literal

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver


**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

In [ ]:
# 2.1 Skill-Dateien aus GitHub laden und lokal cachen

SKILL_DIR = Path("./_skill_pmo_briefing")

copy_from_github(
    source="ralf-42/Agenten/06_skill/pmo-briefing",
    target=str(SKILL_DIR),
)

SCRIPT_PATH = SKILL_DIR / "scripts" / "extract_actions.py"

_file_map = {
    "skill":        SKILL_DIR / "SKILL.md",
    "writer":       SKILL_DIR / "references" / "writer-format.md",
    "agenda_rules": SKILL_DIR / "references" / "agenda_rules.md",
    "action_rules": SKILL_DIR / "references" / "action_rules.md",
    "examples":     SKILL_DIR / "references" / "examples.md",
}

def read_skill(name: str) -> str:
    """Liest eine Skill-Datei aus dem lokalen Cache."""
    if name not in _file_map:
        return f"Unbekannte Datei: {name}. Verfügbar: {list(_file_map.keys())}"
    return _file_map[name].read_text(encoding="utf-8").strip()

mprint(f"Skill-Verzeichnis: `{SKILL_DIR}`")
mprint(f"Skill-Dateien: {list(_file_map.keys())}")
mprint(f"Skript: `{SCRIPT_PATH}`")

In [ ]:
# Der Kontext wird im nächsten Abschnitt aus dem gemeinsamen PMO-RAG abgerufen.
PREPARATION_QUERIES = {
    "beschluesse_2026_05_20": "D1 D2 Vektordatenbank Produktivbetrieb Rechte Rollen Entscheidung vertagt Stand 20.05.2026",
    "risiken_2026_05_20": "R2 Datenschutzfreigabe Testkorpus produktive Kundendaten nächste Schritte Stand 20.05.2026",
}


**Gemeinsamer PMO-Korpus:** Wie in den Harness-Grundlagen wird die bestehende Chroma-Collection aus den RAG-Grundlagen beziehungsweise dem RAG-Agenten auf Drive nur gelesen. Keine lokale Neuindexierung und keine separaten Demo-DOCX-Dateien. Projekt- und Quellenstatusfilter schließen Fremdprojekte, Rohnotizen und überholte Entwürfe aus. Die Nachbereitung zum 28.04.2026 verwendet ausschließlich Fundstellen bis zu diesem Stichtag; spätere Beschlüsse werden nicht rückwirkend eingemischt.


In [ ]:
from genai_lib.pmo_rag import get_pmo_vectorstore, format_docs

# Bestehenden Index aus M11/M12 öffnen; hier keine Neuindexierung durchführen.
vectorstore = get_pmo_vectorstore()
if not vectorstore.get(limit=1)["ids"]:
    raise RuntimeError("PMO-Collection ist leer. Zuerst M11/M12 zur Indexierung ausführen.")

pmo_filter = {"$and": [
    {"project_id": {"$eq": "projekt_kompass"}},
    {"source_status": {"$in": ["approved_record", "approved_current"]}},
]}
if not vectorstore.get(where=pmo_filter, limit=1)["ids"]:
    raise RuntimeError("Keine freigegebenen Kompass-Quellen mit Metadaten. Index in M11/M12 aktualisieren.")
pmo_retriever = vectorstore.as_retriever(search_kwargs={"k": 8, "filter": pmo_filter})
print("Bestehende PMO-Collection geöffnet; Projekt- und Quellenfilter aktiv.")


In [ ]:
import re
from markitdown import MarkItDown
from langchain_core.documents import Document

SOURCE_DIR = Path("./_m51_original_sources")
SOURCE_REPO = "ralf-42/Agenten/02_daten/01_text/korpus_pmo_assistent"
copy_from_github(source=SOURCE_REPO, target=str(SOURCE_DIR), mask="metadata.json")
SOURCE_METADATA = {entry["file"]: entry for entry in
                   json.loads((SOURCE_DIR / "metadata.json").read_text(encoding="utf-8"))["documents"]}
source_converter = MarkItDown()
source_documents = {}

def load_original_source(name: str, cutoff: str):
    """Lädt nur registrierte, freigegebene Kompass-Originale bis zum Stichtag."""
    meta = SOURCE_METADATA.get(name, {})
    if (meta.get("project_id") != "projekt_kompass"
            or meta.get("source_status") not in {"approved_record", "approved_current"}
            or not meta.get("date") or meta["date"] > cutoff):
        return None
    if name not in source_documents:
        copy_from_github(source=SOURCE_REPO, target=str(SOURCE_DIR), mask=name)
        text = source_converter.convert(str(SOURCE_DIR / name)).text_content.strip()
        if not text:
            raise ValueError(f"Leere Originalquelle: {name}")
        source_documents[name] = Document(page_content=text, metadata={**meta, "source": name})
    return source_documents[name]


In [ ]:
def format_full_sources(documents) -> str:
    """Trennt Quellenmetadaten vom vollständigen Dokumenttext, ohne Zeichenlimit."""
    return json.dumps([{"source": doc.metadata["source"], "date": doc.metadata["date"],
                        "text": doc.page_content} for doc in documents], ensure_ascii=False, indent=2)

def retrieve_contexts(queries: dict[str, str], cutoff: str, source_names=None) -> dict[str, str]:
    """RAG wählt Dateien aus; danach werden die vollständigen Originale geladen."""
    originals = {}
    for question in queries.values():
        hits = pmo_retriever.invoke(question)
        for hit in hits:
            name = str(hit.metadata.get("source", "")).replace("\\", "/").rsplit("/", 1)[-1]
            if source_names is not None and name not in source_names:
                continue
            doc = load_original_source(name, cutoff)
            if doc is not None:
                originals[name] = format_full_sources([doc])
    return originals

DEMO_CONTEXTS_BEFORE = retrieve_contexts(PREPARATION_QUERIES, "2026-05-20")
DEMO_CONTEXTS_AFTER = retrieve_contexts({
    "steuerkreis": "Steuerkreis 28.04.2026 D3 Risiko R2 mitigiert Testkorpus nächste Schritte",
}, "2026-04-28", source_names={"protokoll_steuerkreis_2026-04-28.pdf"})
if not DEMO_CONTEXTS_BEFORE or not DEMO_CONTEXTS_AFTER:
    raise RuntimeError("Originalquellen fehlen. RAG-Treffer und vollständigen Index in M11/M12 prüfen.")
print("Vollständige Originalquellen:", len(DEMO_CONTEXTS_BEFORE), "Vorbereitung,", len(DEMO_CONTEXTS_AFTER), "Nachbereitung")


In [ ]:
def context_bundle(context_type="vorbereitung", document_names=None):
    """Löst Dokumentnamen im registrierten Kontext auf; akzeptiert keinen nacherzählten Text."""
    if context_type not in {"vorbereitung", "nachbereitung"}:
        raise ValueError("Unbekannter Kontexttyp.")
    contexts = DEMO_CONTEXTS_BEFORE if context_type == "vorbereitung" else DEMO_CONTEXTS_AFTER
    names = sorted(contexts) if document_names is None else list(dict.fromkeys(document_names))
    if not names or any(name not in contexts for name in names):
        raise ValueError("Nur registrierte Originaldokumente dieses Kontexttyps verwenden.")
    documents = [source_documents[name] for name in names]
    sources = [f"{doc.metadata['source']} — {doc.metadata['date']}" for doc in documents]
    return documents, format_full_sources(documents), sources


In [ ]:
from genai_lib.utilities import load_prompt

# Auszüge aus den lokalen Skill-Dateien (Frontmatter wird automatisch entfernt)
_previews = {
    "SKILL.md":        (str(SKILL_DIR / "SKILL.md"),                        600),
    "agenda_rules.md": (str(SKILL_DIR / "references" / "agenda_rules.md"),  400),
    "WRITER.md":       (str(SKILL_DIR / "references" / "writer-format.md"), 300),
}

lines = []
for name, (path, limit) in _previews.items():
    text = load_prompt(path, mode="S")[:limit]
    lines += [f"### {name} (Auszug)", "", text, ""]

mprint("\n".join(lines))

# 3 | Pydantic-Modelle & Gate-Analyse

---



Damit der Gate-Agent schema-validierten strukturierten Output liefert,
verwenden wir `with_structured_output()` mit Pydantic-Modellen.

| Modell | Aufgaben |
|--------|---------|
| `ActionItemResult` | Einzelnes Action Item (Was / Wer / Wann) |
| `AgendaPointResult` | Agenda-Punkt (Priorität, Ziel, Dauer, Eigentümer) |
| `GateOutputResult` | Vollständiger strukturierter Gate-Output für den Writer |

> 💡 **Warum `with_structured_output()`?**
> Der Writer-Sub-Agent braucht verlässliche Eingabedaten. Ein freier LLM-Text-Output
> kann in Struktur und Vollständigkeit variieren — ein Pydantic-validiertes JSON nicht.

In [ ]:
# 3.1 Pydantic-Modelle für strukturierten Gate-Output

class ActionItemResult(BaseModel):
    """Ein konkretes Action Item aus dem Meeting."""
    task:  str = Field(description="Aufgabenbeschreibung — kurz, aktionsorientiert")
    owner: str = Field(description="Verantwortliche Person — '[offen]' wenn unklar")
    due:   str = Field(description="Explizite Frist aus dem Original, sonst '[offen]'")
    source: str = Field(default="", description="Exakter Dateiname der Originalquelle")
    source_date: str = Field(default="", description="Dokumentdatum, keine Aufgabenfrist")
    evidence: str = Field(default="", description="Wörtlicher Beleg aus dem Originaltext")


class AgendaPointResult(BaseModel):
    """Ein Agenda-Punkt mit Priorität, Ziel und Zeitplan."""
    priority: str = Field(description="Priorität: hoch, mittel oder gering")
    point:    str = Field(description="Bezeichnung des Agenda-Punkts")
    goal:     str = Field(description="Ziel: entscheiden, klären, informieren oder präsentieren")
    duration: int = Field(description="Geplante Dauer in Minuten")
    owner:    str = Field(description="Verantwortliche Person für diesen Punkt")


class GateOutputResult(BaseModel):
    """Strukturierter Gate-Output — Eingabe für den Writer-Sub-Agenten."""
    type:           str                    = Field(description="vorbereitung oder nachbereitung")
    meeting_type:   str                    = Field(description="Projektmeeting, Kundengespräch, Review, Workshop oder Sonstiges")
    topic:          str                    = Field(description="Meeting-Thema")
    date:           Optional[str]          = Field(default=None, description="Datum/Uhrzeit wenn bekannt")
    participants:   List[str]              = Field(description="Teilnehmerliste")
    agenda:         List[AgendaPointResult]      = Field(description="Agenda-Punkte sortiert: hoch → mittel → gering")
    open_questions: List[str]              = Field(description="Offene Fragen vor dem Meeting")
    action_items:   List[ActionItemResult] = Field(description="Action Items aus Kontext und Vorgesprächen")
    risks:          List[str]              = Field(description="Bekannte Risiken und Konflikte")
    status:         str                    = Field(description="ok, no_context oder conflict")
    conflict:       Optional[bool]         = Field(default=False, description="True bei widersprüchlichen Quellinformationen")
    sources:        List[str] = Field(description="Verwendete Quellen mit Dokumentdatum; nur aus dem Kontext")
    release_status: Literal["draft"] = Field(default="draft", description="Fester Status: draft. Menschliche Freigabe bleibt offen; keine automatische Versand- oder Datenschutzfreigabe.")
    release_note: str = Field(default="", description="Freie Erläuterung der Freigabegrenzen; ändert den festen Status draft nicht")
    decisions:      Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: getroffene Entscheidungen")
    action_extraction_status: Literal["completed"] = Field(default="completed", description="Codegesteuert: Aufgabenextraktion vor der Gate-Analyse erfolgreich abgeschlossen")
    action_backlog: List[ActionItemResult] = Field(default_factory=list, description="Weitere belegte Aufgaben")
    open_points:    Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: vertagte Punkte")




In [ ]:
# Gate-Constraint: JUDGE (gpt-6.1-sol) explizit auf Datenextraktion beschränken
_GATE_CONSTRAINT = """
## PFLICHT: Ausgabeformat

Du bist ein Datenextraktor — kein Schreiber.

- **Gib AUSSCHLIESSLICH die strukturierten Felder aus GateOutputResult zurück.**
- Formuliere KEIN PMO-Briefing, keinen Report, keinen Fließtext.
- Schreibe KEINE Tabellen, KEINE Markdown-Sections, KEINEN Abschlusssatz.
- Diese Regel überschreibt alle anderen Anweisungen.
- Übernimm Quellen und Dokumentdatum in sources. Keine Freigabe erteilen.
- Die Aufgabenextraktion wurde vor dieser Analyse im Code erfolgreich ausgeführt.
- Die übergebene Extraktionsbestätigung ist Laufzeitinformation, keine Projektquelle.
- Du brauchst selbst kein Tool aufzurufen. Keine erfundenen Aussagen über fehlende Tool-Aufrufe.
- risks, open_questions und decisions enthalten nur fachliche Projektinformationen, keine internen Ablaufdiagnosen.
- action_extraction_status ist completed.
- release_status ist immer draft. Erläuterungen zu Freigabegrenzen gehören ausschließlich in release_note.
- D1 (Prototyp) und D2 (Produktion) unterscheiden; Testkorpus-Freigabe nicht auf produktive Kundendaten übertragen.
"""

# Gate-Prompt aus GitHub-Skill-Dateien zusammensetzen
_GATE_RULES = (
    read_skill("skill") + "\n\n"
    + read_skill("agenda_rules") + "\n\n"
    + read_skill("action_rules") + "\n\n"
    + _GATE_CONSTRAINT
)

gate_prompt = ChatPromptTemplate([
    ("system", _GATE_RULES),
    ("user", "Meeting-Anfrage: {anfrage}\n\nKontext-Dokumente:\n{kontext}\n\nCodebestätigte Aufgabenextraktion:\n{extraction}"),
])

mprint("✅ **Pydantic-Modelle** definiert: `ActionItemResult`, `AgendaPointResult`, `GateOutputResult`")
mprint("✅ **Gate-Prompt** aus GitHub: `SKILL.md` + `agenda_rules.md` + `action_rules.md`")

In [ ]:
def extract_registered_actions(context_type="vorbereitung", document_names=None):
    """Extrahiert eindeutige Personen-Zuweisungen aus Originaltext, nicht aus Metadaten."""
    documents, _, _ = context_bundle(context_type, document_names)
    people = {person for entry in SOURCE_METADATA.values() for person in entry.get("stakeholders", [])}
    aliases = {name: name for name in people}
    aliases.update({name.removeprefix("Dr. "): name for name in people})
    person_pattern = "|".join(re.escape(name) for name in sorted(aliases, key=len, reverse=True))
    assignment = re.compile(rf"\b(?P<owner>{person_pattern})\s+(?:aktualisiert|erstellt|bereitet|konkretisiert|prüft|liefert|übernimmt|soll|muss|koordiniert|dokumentiert)\b")
    items, seen = [], set()
    for doc in sorted(documents, key=lambda d: d.metadata["date"], reverse=True):
        # Dr.-Abkürzung schützen; Semikola trennen auch zwei Tätigkeiten in einem Satz.
        text = re.sub(r"\s+", " ", doc.page_content).replace("Dr.", "Dr<DOT>")
        sentences = re.split(r";\s*|(?<=[.!?])\s+(?=[A-ZÄÖÜ])", text)
        for sentence in sentences:
            sentence = sentence.replace("Dr<DOT>", "Dr.")
            matches = list(assignment.finditer(sentence))
            for index, match in enumerate(matches):
                end = matches[index + 1].start() if index + 1 < len(matches) else len(sentence)
                evidence = sentence[match.start():end].strip(" ;.")
                if re.search(r"\b(?:erledigt|abgeschlossen)\b", evidence, re.IGNORECASE):
                    continue
                predicate = evidence[len(match.group("owner")):].strip()
                # Modal formulierte Ziele/Fähigkeiten sind keine konkreten Arbeitsaufträge.
                concrete_action = re.search(r"\b(?:prüfen|vorbereiten|erstellen|aktualisieren|liefern|dokumentieren|konkretisieren|koordinieren|implementieren|testen|klären|sammeln|formulieren|skizzieren)\b", predicate, re.IGNORECASE)
                if re.match(r"^(?:soll|muss)\b", predicate, re.IGNORECASE) and not concrete_action:
                    continue
                payload = json.dumps({"text": evidence}, ensure_ascii=False)
                proc = subprocess.run([sys.executable, "-X", "utf8", str(SCRIPT_PATH), payload],
                                      capture_output=True, text=True, encoding="utf-8", check=True)
                candidates = json.loads(proc.stdout).get("action_items", [])
                raw_due = candidates[0]["due"] if candidates else "[offen]"
                # Nur ausdrücklich markierte Fristen übernehmen, nie ein bloßes Dokumentdatum.
                due = raw_due if raw_due.lower().startswith("bis ") and raw_due in evidence else "[offen]"
                deadline = re.search(r"\bbis\s+(?:(?:zum|zur)\s+)?(?:\d{4}-\d{2}-\d{2}|\d{1,2}\.\d{1,2}\.\d{2,4}|nächsten\s+(?:Termin|Steuerkreis)|Montag|Dienstag|Mittwoch|Donnerstag|Freitag|Samstag|Sonntag|morgen|übermorgen|Ende\s+der\s+Woche|Belastungstest|Abnahme|Workshop|Pilotbeginn|Sprint-Ende|Release|Meeting)", evidence, re.IGNORECASE)
                if deadline:
                    due = deadline.group(0)
                owner = aliases[match.group("owner")]
                task = evidence[len(match.group("owner")):].strip()
                if due != "[offen]":
                    task = task.replace(due, "").strip(" ,.")
                key = (task.casefold(), owner, due)
                if key not in seen:
                    seen.add(key)
                    items.append({"task": task, "owner": owner, "due": due, "evidence": evidence,
                                  "source": doc.metadata["source"], "source_date": doc.metadata["date"]})
    # Schulungsregel: bei mehr als zehn Aufgaben fünf anzeigen, Rest als vollständiger Backlog.
    limit = 5 if len(items) > 10 else len(items)
    return {"action_items": items[:limit], "backlog": items[limit:], "backlog_count": len(items[limit:])}


In [ ]:
def has_missing_extraction_claim(result) -> bool:
    """Erkennt negative Ablaufbehauptungen trotz erfolgreich abgeschlossener Extraktion."""
    fields = result.risks + result.open_questions + (result.open_points or []) + (result.decisions or []) + [result.release_note]
    for text in fields:
        technical = re.search(r"Aufgabenextraktion|Aktionspunktextraktion|extract_action_items|Extraktor|Tool[- ]?Aufruf", text, re.IGNORECASE)
        missing = re.search(r"\b(?:nicht|fehlend|fehlt|ausstehend|unterblieben|keine|kein)\b", text, re.IGNORECASE)
        if technical and missing:
            return True
    return False


In [ ]:
@tool
def analyze_meeting_context(request: str, context_type: Literal["vorbereitung", "nachbereitung"] = "vorbereitung",
                            document_names: Optional[List[str]] = None) -> str:
    """Analysiert Originalquellen erst nach erfolgreicher, codebestätigter Aufgabenextraktion."""
    documents, context, sources = context_bundle(context_type, document_names)
    actions = extract_registered_actions(context_type, document_names)
    extraction = json.dumps({"status": "completed", "action_items": actions["action_items"],
                             "action_backlog": actions["backlog"]}, ensure_ascii=False)
    gate_chain = gate_prompt | init_chat_model(JUDGE).with_structured_output(GateOutputResult)
    inputs = {"anfrage": request, "kontext": context, "extraction": extraction}
    for attempt in range(2):
        result = gate_chain.invoke(inputs, config={**run_cfg, "run_name": "m51_kap3_analysiere_meeting_kontext"})
        if not has_missing_extraction_claim(result):
            break
        inputs = {**inputs, "anfrage": request + "\nKorrektur: Aufgabenextraktion ist nachweislich completed. "
                  "Gib ausschließlich fachliche Projektrisiken aus; keine unbelegten internen Ablaufdiagnosen."}
    else:
        raise ValueError("Gate widerspricht dem belegten Extraktionsstatus; Ergebnis wird nicht an den Writer übergeben.")
    result.action_items = [ActionItemResult(**item) for item in actions["action_items"]]
    result.action_backlog = [ActionItemResult(**item) for item in actions["backlog"]]
    result.action_extraction_status = "completed"
    result.sources = sources
    return result.model_dump_json(indent=2)


**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

# 4 | Skill-Tools kapseln

---



Der Koordinator bekommt sieben fokussierte Tools:

| Tool | Quelle | Aufgaben |
|------|--------|---------|
| `list_context_documents` | Notebook | Verfügbare Demo-Dokumente anzeigen |
| `load_context_document` | Notebook | Einzelnes Kontextdokument laden |
| `suche_wissensdatenbank` | **ChromaDB** | Semantische Suche im Projektkorpus (ergänzend zu den Dokument-Tools) |
| `load_skill_asset` | **GitHub** | Skill-Regelwerk zur Laufzeit nachladen |
| `extract_action_items` | **GitHub** (Temp) | Deterministisch via `extract_actions.py` |
| `analyze_meeting_context` | LLM-Chain | JUDGE (gpt-6.1-sol) + `with_structured_output(GateOutputResult)` |
| `get_demo_meeting_request` | Notebook | Demo-Anfrage für Beispiel 1 |

In [ ]:
# 4.1 Tool-Definitionen

@tool
def list_context_documents(type: str = "vorbereitung") -> str:
    """Listet verfügbare Demo-Kontextdokumente für den angegebenen Meeting-Typ.

    Args:
        type: 'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_BEFORE if type == "vorbereitung" else DEMO_CONTEXTS_AFTER
    return json.dumps(sorted(docs.keys()), ensure_ascii=False)


@tool
def load_context_document(name: str, type: str = "vorbereitung") -> str:
    """Lädt ein einzelnes Demo-Kontextdokument.

    Args:
        name: Dateiname (aus list_context_documents)
        type:  'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_BEFORE if type == "vorbereitung" else DEMO_CONTEXTS_AFTER
    if name not in docs:
        return f"Dokument '{name}' nicht gefunden. Verfügbar: {sorted(docs)}"
    return docs[name]


@tool
def load_skill_asset(name: str) -> str:
    """Lädt eine Skill-Datei aus dem GitHub-Repo (SKILL.md, WRITER.md, Regelwerke, Beispiele).

    Erlaubte Werte: skill, writer, agenda_rules, action_rules, examples

    Quelle: github.com/ralf-42/Agenten — 06_skill/pmo-briefing/
    """
    key = name.strip().lower()
    if key not in _file_map:
        return f"Unbekanntes Asset '{name}'. Verfügbar: {list(_file_map.keys())}"
    return read_skill(key)   # aus Cache oder frisch von GitHub




In [ ]:
@tool
def extract_action_items(context_type: Literal["vorbereitung", "nachbereitung"] = "vorbereitung",
                         document_names: Optional[List[str]] = None) -> str:
    """Liefert belegte Personen-Zuweisungen aus registrierten Originaldokumenten samt Quelle und Frist."""
    return json.dumps(extract_registered_actions(context_type, document_names), ensure_ascii=False, indent=2)

@tool
def get_demo_meeting_request() -> str:
    """Liefert die Demo-Meeting-Anfrage für Beispiel 1 (Vorbereitung des nächsten Steuerkreises)."""
    return dedent("""
        Thema: Nächster Steuerkreis Projekt Kompass
        Typ: Vorbereitung
        Datum/Zeit: [offen]
        Teilnehmer: Mara Vogt (Projektleitung), Jonas Reiter (IT-Architektur), Dr. Thomas Brandt (Datenschutz/Compliance), Elif Kaya (Fachbereich Vertrieb), Sabine Holler (Sponsorin)
        Dauer: [offen]
        Ziel: PMO-Briefing zur vertagten Entscheidung über die Rechte- und Rollensteuerung vorbereiten.
    """).strip()


# Semantische Suche im Projektkorpus (ChromaDB aus Abschnitt 2.3) — ergänzt die Dokument-Tools


In [ ]:
@tool("suche_wissensdatenbank")
def knowledge_base_tool(frage: str, context_type: Literal["vorbereitung", "nachbereitung"] = "vorbereitung") -> str:
    """RAG wählt weitere Originaldateien aus und registriert sie im jeweiligen Kontext."""
    cutoff = "2026-05-20" if context_type == "vorbereitung" else "2026-04-28"
    contexts = DEMO_CONTEXTS_BEFORE if context_type == "vorbereitung" else DEMO_CONTEXTS_AFTER
    found = retrieve_contexts({"recherche": frage}, cutoff)
    contexts.update(found)
    return json.dumps(found, ensure_ascii=False) if found else "Nicht im Korpus – in den abgerufenen Quellen nicht belegt."

ALL_TOOLS = [get_demo_meeting_request, list_context_documents, load_context_document,
             knowledge_base_tool, load_skill_asset, extract_action_items, analyze_meeting_context]
print("PMO-Tools:", [t.name for t in ALL_TOOLS])


In [ ]:
# Smoke-Test: direkte Originaldokumente statt nacherzähltem Kontext verwenden.
import pandas as pd
raw = extract_action_items.invoke({"context_type": "vorbereitung"},
                                 config={**run_cfg, "run_name": "m51_kap4_aktionspunkte_extrahieren_raw"})
action_result = json.loads(raw)
df_items = pd.DataFrame(action_result["action_items"], columns=["task", "owner", "due", "source", "source_date"])
display(df_items)
print("Weitere belegte Aufgaben im Backlog:", action_result["backlog_count"])


**Kontext- und Aufgabenregel:** RAG dient hier der Dateiauswahl. Danach werden vollständige Originaltexte ohne das Vorschau-Limit von `format_docs()` geladen. Das Original-Dokumentdatum stammt aus dem Korpusmanifest. Der gemeinsame Chroma-Index wird weiterhin nur gelesen.

Die Aufgabenextraktion ist bewusst konservativ: Sie übernimmt eindeutige Zuweisungen an im Korpus benannte Personen und verwendet das vorhandene Skript zur Kandidatenextraktion. Eindeutige Zuweisungen mit Verben wie „aktualisiert“, „konkretisiert“ oder „bereitet … vor“ ergänzt der Wrapper, weil sie im bisherigen Skript nicht vollständig erkannt werden. Mehrere Personen-Zuweisungen werden getrennt. Bloße Dokumentdaten sind keine Fristen. Nicht erkannte oder implizite Aufgaben benötigen menschliche Nachprüfung; der Extraktor ist keine vollständige semantische Analyse. Das Gate übernimmt geprüfte Aufgaben und die Quellenliste aus dem Code, der Writer darf sie nicht umdeuten.

**Render-Regel:** Der Writer erstellt den Briefing-Text und die Hauptaufgaben. Backlog und Quellenverzeichnis werden genau einmal aus dem validierten Gate-Ergebnis ergänzt. Zusätzliche Quellenblöcke des Writers werden entfernt; Quellenbelege im Fließtext bleiben erhalten.

**Ablauf- und Aufgabentypen:** Die Extraktion läuft vor dem Gate. Ihr erfolgreicher Status wird im Code bestätigt und dem Gate ausdrücklich übergeben. Negative Ablaufbehauptungen werden vor der Writer-Übergabe beanstandet. Modal formulierte Projektziele ohne konkretes Arbeitsverb werden nicht als Aufgaben übernommen; ausdrücklich genannte Meilensteinfristen bleiben als relative Frist erhalten.


<font color="black" size="5">Best-Practice-Hinweise für dieses Beispiel</font>

- Die gepinnte Harness-Version bleibt erhalten; Skill-Dateien und Skript liegen im lokalen Cache.
- Der gemeinsame PMO-Index aus M11/M12 wird nur gelesen. Abruf, Formatierung und Gate-Auswertung bleiben getrennt.
- Das Gate validiert die Struktur mit Pydantic. Das garantiert weder deterministische Modellinhalte noch fachliche Richtigkeit.
- `sources` und `release_status` werden vollständig an den Writer weitergegeben. Isolierte Sub-Agenten sehen den Kontext des Koordinators nicht automatisch.
- Der Writer verwendet explizit `WORKER`; eine Ausgabe ist ein Entwurf mit offener menschlicher Freigabe.

**Schema-Regel:** Maschinenprüfbare Zustände werden als feste Werte typisiert (`release_status: Literal["draft"]`). Freie Erläuterungen stehen getrennt in `release_note`. Ein Wortlautvergleich über frei generierten Text ist keine zuverlässige Zustandsprüfung.


# 5 | DeepAgent aufbauen

---

In [ ]:
# 5.1 Writer Sub-Agent — System-Prompt via load_prompt() aus lokalem Cache (references/writer-format.md)
writer_subagent = {
    "name": "briefing-writer",
    "description": "Formatiert den Gate-Output exakt im Ausgabeformat des PMO-Briefing-Skills.",
    "model": init_chat_model(WORKER),   # gpt-6.1-terra — Textqualität für Briefings
    "system_prompt": (
        load_prompt(str(SKILL_DIR / "references" / "writer-format.md"), mode="S")   # lokal aus Cache
        + "\n\n"
        + "Gib ausschließlich das fertige Briefing aus.\n"
        + "Keine Websuche, keine neuen Fakten, keine Spekulation. Erhalte sources und Dokumentdatum. release_status=draft bedeutet: Entwurf – menschliche Freigabe offen. Übernimm Erläuterungen aus release_note, ohne daraus eine Freigabe abzuleiten. Übernimm action_items unverändert samt Quellen und Fristen. Gib keinen eigenen Backlog aus; das Notebook ergänzt action_backlog vollständig. Erfinde keine zusätzlichen Dokumentnamen oder Toolnamen als Quellen. Gib kein eigenes Quellenverzeichnis aus; das Notebook ergänzt es aus den validierten sources. action_extraction_status=completed ist der verbindliche Ausführungsnachweis. Berichte keine hypothetischen internen Tool-Fehler als Projektrisiken. Agenda-Zeitvorschläge als Vorschlag kennzeichnen."
    ),
    "tools": [],
}



In [ ]:
# 5.2 DeepAgent Koordinator
COORDINATOR_PROMPT = (
    "Du bist der Koordinator für den technischen Skill `pmo-briefing`.\n\n"
    "Pflicht-Ablauf:\n"
    "1. get_demo_meeting_request aufrufen (Beispiel 1) ODER Anfrage aus Nachricht lesen\n"
    "2. Übergebenen Kontext vollständig lesen; falls keiner übergeben wurde, list_context_documents mit dem passenden Typ aufrufen und alle dort gelisteten Kontextpakete laden\n"
    "   Optional: Für frühere Entscheidungen, Risiken oder offene Punkte suche_wissensdatenbank mit dem passenden context_type nutzen (ergänzt das Laden, ersetzt es nicht)\n"
    "3. extract_action_items mit context_type und den exakten document_names aufrufen; keinen Kontext nacherzählen\n"
    "4. analyze_meeting_context mit request, demselben context_type und denselben document_names aufrufen\n"
    "5. Vollständiges GateOutputResult einschließlich sources, release_status und release_note sowie den Extraktor-Output an briefing-writer delegieren\n\n"
    "Hard Rules:\n"
    "- IMMER extract_action_items aufrufen; ausschließlich registrierte Dateinamen übergeben, nie paraphrasierten Text\n"
    "- IMMER analyze_meeting_context aufrufen — kein freies Briefing-Schreiben\n"
    "- Treffer aus suche_wissensdatenbank nur mit Quellenangabe verwenden; Quellenrang und Datum prüfen; neuer Beschluss ersetzt historischen Stand nur in seinem Geltungsbereich\n"
    "- Fehlende Angaben als [offen] markieren, nie spekulieren\n"
    "- Keine Freigabe oder Versandaktion. Dokumentdatum, Fallangabe und offene Information unterscheiden\n"
    "- Nur Quellen aus GateOutputResult.sources nennen; Recherche-Tools sind keine Quelldokumente\n"
    "- Antwort auf Deutsch"
)

meeting_agent = create_deep_agent(
    model=init_chat_model(ROUTER, reasoning_effort="none"),   # Koordinator → gpt-6-luna (Tools → reasoning_effort="none")
    tools=ALL_TOOLS,
    subagents=[writer_subagent],
    system_prompt=COORDINATOR_PROMPT,
    checkpointer=InMemorySaver(),
)

mprint("✅ **DeepAgent** erstellt")
mprint(f"   Koordinator: `gpt-6-luna`  ·  Sub-Agent: `briefing-writer`  ·  Tools: {len(ALL_TOOLS)}")

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

<p><font color='black' size="5">
5.3 HarnessProfile — Modell-spezifische Optimierung
</font></p>

DeepAgents liefert **eingebaute Profile** für OpenAI und Anthropic.
Sie aktivieren automatisch und verbessern die Agent-Qualität laut Benchmark um 10–20 Punkte — ohne Code-Änderungen.

**Wann ein eigenes `HarnessProfile` sinnvoll ist:**

| Anwendungsfall | Beispiel |
|---------------|---------|
| Sprachvorgabe | Deutsch erzwingen, auch wenn Skill-Dateien auf Englisch sind |
| Skill-weiter Prompt-Suffix | Ausgabeformat für alle Agenten in diesem Notebook einheitlich |
| Provider-Override | Unterschiedliche Einstellungen für OpenAI vs. Anthropic |

Das Profil wird **einmalig registriert** (`register_harness_profile()`) und wirkt auf alle nachfolgenden `create_deep_agent()`-Aufrufe in dieser Session.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf im Koordinator</font> </br></p>


# Sequenzdiagramm: Ablauf im Koordinator
diagram_seq = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
sequenceDiagram
    autonumber
    actor N as Nutzer
    participant K as Koordinator-Agent
    participant L as LLM (JUDGE · gpt-6.1-sol)
    participant GH as GitHub Repo
    participant D as Dokument-Tools
    participant V as Wissensdatenbank (ChromaDB)
    participant G as analyze_meeting_context
    participant E as extract_action_items
    participant W as briefing-writer

    Note over K,GH: Setup (einmalig)
    GH-->>K: SKILL.md, WRITER.md, Regelwerke, extract_actions.py

    N->>K: Meeting-Anfrage
    K->>L: Anfrage + Tool-Beschreibungen

    loop Solange das LLM Tools aufruft
        L-->>K: Tool-Call
        Note over K,W: Der Koordinator-Agent führt die Tools aus,<br/>nicht das LLM
        alt Dokumente laden
            K->>D: list_context_documents()
            D-->>K: ["doc1.md", "doc2.txt", ...]
            K->>D: load_context_document(name) [x n]
            D-->>K: Dokumentinhalte
        else Korpus durchsuchen (ergänzend)
            K->>V: suche_wissensdatenbank(frage)
            V-->>K: Treffer mit Quellen
        else Action Items extrahieren
            K->>E: extract_action_items(text)
            E-->>K: Action Items JSON
        else Kontext analysieren
            K->>G: analyze_meeting_context(request, context)
            G-->>K: GateOutputResult JSON (Pydantic-validiert)
        else Briefing schreiben
            K->>W: GateOutputResult übergeben
            W-->>K: Formatiertes Briefing
        end
        K->>L: Tool-Ergebnis + bisheriger Kontext
    end

    L-->>K: Finale Antwort
    K-->>N: PMO-Briefing
'''
mermaid(diagram_seq, width=1200)

In [ ]:
# -- VIZ --
from IPython.display import Image, display

display(Image(meeting_agent.get_graph(xray=True).draw_mermaid_png()))

# 6 | Beispiel-Runs

---

| Beispiel | Typ | Szenario |
|----------|-----|----------|
| 1️⃣ | Vorbereitung | Nächster Steuerkreis Projekt Kompass — via Demo-Dokument-Tools |
| 2️⃣ | Nachbereitung | Steuerkreis vom 28.04.2026 — Protokollauszug direkt übergeben |

In [ ]:
import re as _re

def invoke_with_retry(agent, inputs, config, max_retries: int = 5):
    """Wiederholt agent.invoke() bei RateLimitError mit automatischer Wartezeit."""
    for attempt in range(1, max_retries + 1):
        try:
            return agent.invoke(inputs, config=config)
        except Exception as e:
            msg = str(e)
            if "rate_limit_exceeded" not in msg and "429" not in msg:
                raise
            match = _re.search(r"try again in ([\d.]+)s", msg)
            wait = float(match.group(1)) + 2 if match else 15
            print(f"Rate limit (Versuch {attempt}/{max_retries}) — warte {wait:.0f}s …")
            time.sleep(wait)
    raise RuntimeError(f"Rate limit nach {max_retries} Versuchen nicht behoben.")

In [ ]:
def strip_writer_reference_sections(text: str) -> str:
    """Entfernt eigene Quellen- und Backlog-Blöcke; Belege im Fließtext bleiben erhalten."""
    lines, skip_depth, skip_paragraph = [], None, False
    for line in text.splitlines():
        heading = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)
        if skip_depth is not None:
            if heading and len(heading.group(1)) <= skip_depth:
                skip_depth = None
            else:
                continue
        if skip_paragraph:
            if not line.strip() or heading:
                skip_paragraph = False
            else:
                continue
        if heading and re.match(r"(?:Quellen?(?:verzeichnis)?|Sources|Backlog)\b", heading.group(2), re.IGNORECASE):
            skip_depth = len(heading.group(1))
            continue
        if re.match(r"^\s*\*\*(?:Quellen?(?:verzeichnis)?|Sources|Backlog)\b[^*]*\*\*", line, re.IGNORECASE):
            skip_paragraph = True
            continue
        lines.append(line)
    return "\n".join(lines).strip()

def markdown_cell(value: str) -> str:
    """Schützt Tabellenzellen vor zusätzlichen Spalten und Zeilen."""
    return str(value).replace("|", "\\|").replace("\n", " ")


In [ ]:
def render_briefing(result) -> str:
    """Erzeugt Backlog und einziges Quellenverzeichnis aus dem validierten Gate-Ergebnis."""
    gates = [message for message in result["messages"]
             if getattr(message, "type", "") == "tool" and message.name == "analyze_meeting_context"]
    gate = GateOutputResult.model_validate_json(gates[-1].content)
    body = strip_writer_reference_sections(result["messages"][-1].text)
    sections = [body]
    if gate.action_backlog:
        rows = [f"### Backlog ({len(gate.action_backlog)} weitere Aufgaben)", "",
                "| Aufgabe | Verantwortlich | Frist | Quelle / Dokumentdatum |",
                "|---|---|---|---|"]
        for item in gate.action_backlog:
            values = [item.task, item.owner, item.due, f"{item.source} — {item.source_date}"]
            rows.append("| " + " | ".join(markdown_cell(value) for value in values) + " |")
        sections.append("\n".join(rows))
    sections.append("### Quellen und Dokumentdaten\n\n" +
                    "\n".join(f"- {source}" for source in dict.fromkeys(gate.sources)))
    sections.append("**Status des Ergebnisses:** Entwurf – menschliche Freigabe offen.")
    return "\n\n".join(sections)


In [ ]:
with tracing_context(enabled=True):
    # 6.1 Beispiel: Vorbereitung des nächsten Steuerkreises
    task_1 = dedent("""
        Bereite das Meeting vor und wende den technischen Skill `pmo-briefing` korrekt an.
        Projekt Kompass, dokumentierter Stand 20.05.2026. Nutze alle verfügbaren RAG-Kontextpakete (typ='vorbereitung'),
        extrahiere Action Items mit dem Tool und gib am Ende nur den Briefing-Entwurf mit Quellen, Dokumentdatum und offener menschlicher Freigabe aus.
    """).strip()

    print(task_1)
    print()

    t0 = time.perf_counter()
    result_1 = invoke_with_retry(
        meeting_agent,
        {"messages": [HumanMessage(content=task_1)]},
        config={
            "recursion_limit": 120,
            "configurable": {"thread_id": "vorbereitung"},
            "run_name": "m51_kap6_briefing_vorbereitung",
            "tags": ["deepagents", "skill", "vorbereitung"],
        },
    )
    latency_1 = round((time.perf_counter() - t0) * 1000)

    mprint("\n".join([
        "## Laufmetriken Beispiel 1",
        "",
        "| Metrik | Wert |",
        "|--------|------|",
        f"| Nachrichten gesamt | `{len(result_1['messages'])}` |",
        f"| Latenz             | `{latency_1} ms` |",
        f"| Letzte Rolle       | `{result_1['messages'][-1].type}` |",
    ]))

In [ ]:
briefing_1 = render_briefing(result_1)
mprint(briefing_1)

In [ ]:
with tracing_context(enabled=True):
    # 6.2 Beispiel: Steuerkreis-Nachbereitung
    context_after = "\n\n".join(
        [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_AFTER.items()]
    )

    task_2 = dedent(f"""
        Erstelle die Nachbereitung für den folgenden Steuerkreis.
        Wende den technischen Skill `pmo-briefing` korrekt an und extrahiere
        Action Items mit dem Tool. Gib am Ende nur den Briefing-Entwurf mit Quellen, Dokumentdatum und offener menschlicher Freigabe aus.
        Verwende ausschließlich den übergebenen historischen Kontext bis 28.04.2026.

        Meeting-Anfrage:
        - Thema: Steuerkreis Projekt Kompass: Risikoliste-Update
        - Datum: 2026-04-28
        - Teilnehmer: Mara Vogt, Dr. Thomas Brandt, Jonas Reiter
        - Typ: Nachbereitung

        Kontextdokumente:
        {context_after}
    """).strip()

    t0 = time.perf_counter()
    result_2 = invoke_with_retry(
        meeting_agent,
        {"messages": [HumanMessage(content=task_2)]},
        config={
            "recursion_limit": 120,
            "configurable": {"thread_id": "nachbereitung"},
            "run_name": "m51_kap6_briefing_nachbereitung",
            "tags": ["deepagents", "skill", "nachbereitung"],
        },
    )
    latency_2 = round((time.perf_counter() - t0) * 1000)

    mprint("\n".join([
        "## Laufmetriken Beispiel 2",
        "",
        "| Metrik | Wert |",
        "|--------|------|",
        f"| Nachrichten gesamt | `{len(result_2['messages'])}` |",
        f"| Latenz             | `{latency_2} ms` |",
        f"| Letzte Rolle       | `{result_2['messages'][-1].type}` |",
    ]))



In [ ]:
briefing_2 = render_briefing(result_2)
mprint(briefing_2)


In [ ]:
# Tatsächliche Tool-Ergebnisse prüfen; eine fertige Antwort allein ist kein Nachweis.
for label, result in [("vorbereitung", result_1), ("nachbereitung", result_2)]:
    calls = [call["name"] for message in result["messages"]
             for call in getattr(message, "tool_calls", [])]
    assert {"extract_action_items", "analyze_meeting_context"} <= set(calls), (label, calls)
    outputs = [message for message in result["messages"]
               if getattr(message, "type", "") == "tool" and message.name == "analyze_meeting_context"]
    assert outputs, f"Kein Gate-Ergebnis für {label}"
    gate_result = GateOutputResult.model_validate_json(outputs[-1].content)
    assert gate_result.sources, f"Quellen fehlen für {label}"
    assert gate_result.release_status == "draft", gate_result.release_status
    assert gate_result.action_extraction_status == "completed"
    assert not has_missing_extraction_claim(gate_result), "Widersprüchliche Ablaufdiagnose im Gate"
    allowed_sources = {f"{name} — {doc.metadata['date']}" for name, doc in source_documents.items()}
    assert set(gate_result.sources) <= allowed_sources, gate_result.sources
    for item in gate_result.action_items + gate_result.action_backlog:
        doc = source_documents[item.source]
        assert item.source_date == doc.metadata["date"]
        assert " ".join(item.evidence.split()) in " ".join(doc.page_content.split())
    print(label, "Gate-Struktur, Quellenliste und offener Freigabestatus geprüft.")

In [ ]:
#@markdown   <p><font size="4" color="green">Trace-Analyse</font></p>

time.sleep(2)
show_trace(os.environ["LANGSMITH_PROJECT"], limit=3, show_steps=True)

# 7 | Einordnung

---

<font color="black" size="5">Was an diesem Beispiel wichtig ist</font>

**Skill-Dateien aus dem Repo — nicht im Notebook**

Prompts, Regeln und Skripte kommen aus `github.com/ralf-42/Agenten`.
Das Notebook ist der Ausführungskontext, nicht die Wissensquelle.
Ändert sich `SKILL.md` im Repo, übernimmt das Notebook die neue Version
beim nächsten Run automatisch — ohne manuelles Copy-Paste.

**Structured Output schützt den Writer**

`with_structured_output(GateOutputResult)` stellt sicher, dass der Writer-Sub-Agent
immer vollständige, typisierte Eingaben erhält — unabhängig davon, wie der
Koordinator die Analyse formuliert.

**Grenzen des Musters**

| Grenze | Erklärung |
|--------|-----------|
| Writer-Modell | Explizit `WORKER`; Quellen und Gate-Ergebnis müssen trotzdem vollständig übergeben werden |
| Kein HITL | Für Produktion: `interrupt()` vor Briefing-Versand ergänzen |
| Kein Judge | Skill-Compliance nicht automatisch geprüft |
| Kein Auth | GitHub-Zugriff anonym — für private Repos: Token-Authentifizierung ergänzen |
| Keine autonome PMO-Steuerung | Status und Empfehlungen bleiben quellengebunden; verbindliche Entscheidungen und Aktionen benötigen menschliche Freigabe |

# 8 | Skills für Arbeitsstandards
---


Ein verbreitetes Missverständnis: Skills müssen keine Algorithmen enthalten. Sie können **Stil, Regeln und Qualitätsvorgaben operationalisieren** — alles, was sonst in langen System-Prompts oder unstrukturierten Dokumenten verstreut ist.

Das PMO-Briefing in diesem Modul ist selbst ein Beispiel dafür: Es enthält keine Berechnungen, sondern **Arbeitsstandards**:
- Feste Abschnitte (Agenda, Vorabinfo, Offene Punkte, Nächste Schritte)
- Pflichtfelder mit `[offen]`-Markierung statt Weglassen
- Quellenpflicht für jede Aussage
- Ton- und Formatvorgaben über `writer-format.md`

**Weitere Anwendungsfälle für "Standards-Skills":**

| Typ | Was der Skill operationalisiert | Beispiel |
|-----|--------------------------------|---------|
| Brand Skill | Farben, Typografie, Logoregeln | CraftedWell-Stil für PPT/DOCX |
| Schreibstil | Ton, Formulierungsverbote, Zielgruppe | "Immer Du, keine Passivkonstruktionen" |
| Qualitätsprüfung | Checklisten, Mindestanforderungen | Review-Kriterien vor Veröffentlichung |
| Onboarding-Prozess | Reihenfolge, Pflichtschritte, Eskalation | Briefing-Freigabe-Check |
| Genehmigungsworkflow | Schwellenwerte, Eskalationsstufen | Budgetfreigabe über bestimmtem Betrag |

> 💡 **Kerngedanke:** Ein Skill macht implizites Organisationswissen explizit — und damit für einen Agenten nutzbar. Nicht "was kann das Modell", sondern "was weiß unsere Organisation und wie wird es angewendet".


# 9 | Skill-Qualität und Skill-Tests
---


Wie prüft man, ob ein Skill zuverlässig funktioniert? Die Antwort ist analog zu Unit-Tests in der Softwareentwicklung: **kontrollierte Eingaben mit erwartbarem Verhalten**.

<p><font color='black' size="5">
Grundstruktur eines Skill-Tests
</font></p>


```
tests/
├── test_cases.yaml          # Testfälle mit Eingabe + expected_behavior
├── input/                   # Beispieldokumente als Eingabe
│   ├── steuerkreis_vorbereitung.md
│   └── steuerkreis_notizen.txt
└── expected/                # Referenzausgaben (optional, für strukturierte Formate)
    └── briefing_example.md
```

Ein Testfall (YAML-Format):

```yaml
- name: "Steuerkreis-Vorbereitung mit vollständigen Infos"
  skill: pmo-briefing
  input_files:
    - input/steuerkreis_vorbereitung.md
  prompt: "Bereite das PMO-Briefing für den nächsten Steuerkreis vor."
  expected_behavior: |
    - Abschnitt 'Agenda' vorhanden
    - Abschnitt 'Offene Punkte' vorhanden
    - Keine Spekulationen; fehlende Infos als [offen] markiert
    - Maximal 300 Wörter

- name: "Fehlende Teilnehmerinfos"
  skill: pmo-briefing
  input_files:
    - input/steuerkreis_notizen.txt
  prompt: "Erstelle das Briefing für den Steuerkreis."
  expected_behavior: |
    - Teilnehmer-Abschnitt enthält [offen]-Marker
    - Kein Erfinden von Namen oder Rollen
```


<p><font color='black' size="5">
Drei Qualitätsdimensionen
</font></p>

| Dimension | Frage | Test-Ansatz |
|-----------|-------|-------------|
| **Vollständigkeit** | Sind alle Pflichtabschnitte vorhanden? | Regex / Strukturprüfung |
| **Korrektheit** | Werden nur Fakten aus dem Input verwendet? | LLM-as-Judge |
| **Konsistenz** | Liefert der Skill bei gleicher Eingabe ähnliche Outputs? | Mehrfachausführung + Vergleich |


<p><font color='black' size="5">
Wann sind Skill-Tests besonders wichtig?
</font></p>

- Bei Skills mit **Guardrails** (was darf nicht weggelassen werden)
- Bei **mehrstufigen Workflows** mit Pflichtschritten
- Nach jeder Änderung an `SKILL.md` oder `references/`
- Vor dem **Einsetzen in Production**

> ⚠️ **Faustregel:** Wenn ein Skill in einer realen Entscheidung (Compliance, Freigabe, Kundenkommunikation) eingesetzt wird — dann braucht er Tests, bevor er deployed wird.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Skill-Test Qualitäts-Dimensionen</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    IN["Eingabe
(Dokumente + Prompt)"]
    SK["Skill ausführen
(Agent + SKILL.md)"]
    OUT["Output
(Briefing / Ergebnis)"]
    subgraph Checks ["Qualitätsprüfung"]
        C1["Vollständigkeit
Pflichtabschnitte vorhanden?"]
        C2["Korrektheit
Nur Fakten aus Input?"]
        C3["Konsistenz
Wiederholbar bei gleicher Eingabe?"]
    end
    IN --> SK --> OUT --> Checks
    C1 -->|"✅ / ❌"| Report["Test-Report"]
    C2 -->|"✅ / ❌"| Report
    C3 -->|"✅ / ❌"| Report

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class IN input;
    class C1,C2,C3 metric;
    class OUT,Report output;
'''
mermaid(diagram, width=1050)


# A | Aufgaben

---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen** Erstelle aus den vorhandenen RAG-Fundstellen einen Kompass-Briefing-Entwurf für Mara Vogt zu D2, R2 und offenen Zugriffsregeln. Prüfe Pflichtfelder, Quellen mit Datum und den offenen Freigabestatus. Verwende keine erfundenen Meeting-Termine.


In [ ]:
# Grundlagen: eigenes Meeting mit vorhandenen Kompass-Originalquellen.
my_meeting = "Steuerkreis Projekt Kompass – Pilotvoraussetzungen; Datum und Dauer offen"
# 1. Dokumentnamen aus list_context_documents(type="vorbereitung") wählen.
# 2. meeting_agent mit my_meeting, context_type="vorbereitung" und diesen Namen aufrufen.
# 3. Quellen, action_items, action_backlog und offenen Freigabestatus prüfen.


**Aufbau** Teste fehlende oder unzulässige Kontextabdeckung. Eine nicht registrierte Datei muss abgewiesen werden, statt frei formulierten Ersatztext zu akzeptieren. Frage anschließend anhand der Originalquellen nach monatlichen Betriebskosten nach dem Kompass-Rollout. Das Gate muss fehlende Evidenz benennen; aus dem Q2-Forecast dürfen keine Monatskosten entstehen.


In [ ]:
# Aufbau: fehlende Kontextabdeckung testen.
# analyze_meeting_context verwendet registrierte Originale und akzeptiert keinen freien Ersatztext.
# 1. Eine nicht registrierte Datei über document_names übergeben: ValueError erwartet.
# 2. Nach nicht belegten monatlichen Betriebskosten fragen.
# 3. Gate-Antwort auf fehlende Evidenz prüfen; Q2-Forecast darf keine Monatskosten ersetzen.


**Vertiefung** Prüfe zuerst D1 (ChromaDB für Prototypen) und D2 (Cloud für Produktion): Unterschiedlicher Zeitpunkt und Geltungsbereich sind kein ungelöster Widerspruch. Teste danach zwei ausdrücklich synthetische Beschlüsse mit gleichem Datum und Geltungsbereich, die unterschiedliche Produktivlösungen verlangen. Der LCEL-Judge muss diesen echten Konflikt erkennen und Quellen sowie Datum nennen.


In [ ]:
# Vertiefung: Konflikt-Szenario + LCEL-Judge
# Startpunkt: LLM-as-Judge

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class ConflictResult(BaseModel):
    conflict_found: bool
    description: str = Field(description='Was widerspricht sich?')
    severity: str = Field(description='niedrig / mittel / hoch')

# doc_a = 'Steuerkreis 17.03.2026 (D1): ChromaDB wird für den Prototyp evaluiert.'
# doc_b = 'Steuerkreis 14.04.2026 (D2): Verwaltete Cloud-Vektordatenbank für den produktiven Einsatz; ChromaDB verworfen.'

# judge_llm = init_chat_model(JUDGE)  # JUDGE = gpt-6.1-sol
# judge = judge_llm.with_structured_output(ConflictResult)
# urteil = judge.invoke(f'Dokument A: {doc_a}\nDokument B: {doc_b}', config=run_cfg)
# print(urteil)

<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Skills](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/skills.html)
- [Einsteiger DeepAgents](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-deepagents.html)
- [PMO-Assistent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/pmo-assistent-praxis.html)
- [Multi-Agent-Systeme](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/multi-agent-systeme.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
